In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "phase1_data" else Path.cwd()
posts = pd.read_csv(ROOT / "data" / "interim" / "clean_posts.csv", dtype={"text": str}, encoding="utf-8-sig")
posts["text"] = posts["text"].fillna("")

own = posts[posts["copied"] == 0]
docs = own.groupby("patient")["text"].apply(lambda t: " ".join(t)).rename("doc").reset_index()
docs = docs.merge(posts.drop_duplicates("patient")[["patient", "label"]], on="patient")

test_ids = set(pd.read_csv(ROOT / "data" / "processed" / "test_patients.csv")["patient"])
docs["split"] = docs["patient"].map(lambda p: "test" if p in test_ids else "train")
print(docs.groupby(["split", "label"]).size().unstack())

label   0    1
split         
test   12   30
train  49  118


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

def text_model():
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=20000, sublinear_tf=True),
        LogisticRegression(C=1.0, class_weight="balanced", max_iter=3000),
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tr = docs[docs["split"] == "train"].reset_index(drop=True)

oof = cross_val_predict(text_model(), tr["doc"], tr["label"], cv=cv, method="predict_proba")[:, 1]
print("Text alone, cross-validated AUC:", round(roc_auc_score(tr["label"], oof), 3))

Text alone, cross-validated AUC: 0.617


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score

feat = pd.read_csv(ROOT / "data" / "processed" / "features_train.csv")
data = feat.merge(tr[["patient", "doc"]], on="patient")
Xn, y, D = data.drop(columns=["patient", "label", "doc"]), data["label"], data["doc"]

def numeric_model(k):
    return make_pipeline(StandardScaler(), SelectKBest(f_classif, k=k),
                         SVC(kernel="linear", C=0.1, class_weight="balanced", probability=True, random_state=42))

res = {"numbers only": [], "numbers + text": []}
for tr_i, va_i in cv.split(Xn, y):
    # text score for the training part (inner out-of-fold) and the validation part
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    ts_tr = cross_val_predict(text_model(), D.iloc[tr_i], y.iloc[tr_i], cv=inner, method="predict_proba")[:, 1]
    ts_va = text_model().fit(D.iloc[tr_i], y.iloc[tr_i]).predict_proba(D.iloc[va_i])[:, 1]

    m1 = numeric_model(15).fit(Xn.iloc[tr_i], y.iloc[tr_i])
    p1 = m1.predict_proba(Xn.iloc[va_i])[:, 1]

    Xtr2 = Xn.iloc[tr_i].assign(text_score=ts_tr); Xva2 = Xn.iloc[va_i].assign(text_score=ts_va)
    m2 = numeric_model(16).fit(Xtr2, y.iloc[tr_i])
    p2 = m2.predict_proba(Xva2)[:, 1]

    res["numbers only"].append(roc_auc_score(y.iloc[va_i], p1))
    res["numbers + text"].append(roc_auc_score(y.iloc[va_i], p2))

for k, v in res.items():
    print(f"{k:15s} AUC {np.mean(v):.3f} ± {np.std(v):.3f}")

numbers only    AUC 0.683 ± 0.071
numbers + text  AUC 0.663 ± 0.074


In [4]:
te = docs[docs["split"] == "test"].reset_index(drop=True)
tr_score = pd.DataFrame({"patient": tr["patient"], "text_score": oof})
te_score = pd.DataFrame({"patient": te["patient"],
                         "text_score": text_model().fit(tr["doc"], tr["label"]).predict_proba(te["doc"])[:, 1]})

PROC = ROOT / "data" / "processed"
for name, sc in [("train", tr_score), ("test", te_score)]:
    f = pd.read_csv(PROC / f"features_{name}.csv").merge(sc, on="patient")
    cols = [c for c in f.columns if c != "label"] + ["label"]
    f[cols].to_csv(PROC / f"features_{name}_v2.csv", index=False)
print("Saved features_train_v2.csv and features_test_v2.csv (24 features)")

Saved features_train_v2.csv and features_test_v2.csv (24 features)
